# NoFluffJobs API search

Query the NoFluffJobs posting search endpoint, inspect the response, and collect the returned postings into a clean list.

## 1. Imports

We use `httpx` for the HTTP request and `pandas` later for tabular display.

In [28]:
import httpx
import pandas as pd

## 2. Endpoint and query parameters

`params` are sent as query string arguments. Salary is requested in EUR per month. The `region`/`requirement` filters are left commented so the search stays broad.

In [29]:
url = "https://nofluffjobs.com/api/search/posting"

# Query parameters sent on the URL
params = {
    "salaryCurrency": "EUR",
    "salaryPeriod": "month",
    # "region": "nl",          # 'nl', 'pl', 'cz', 'hu', etc.
    "language": "en",
    "page": 1,
}

## 3. Request body

The backend expects the key `criteriaSearch` (not `criteria`). Add filters such as `requirement` inside it to narrow the search.

In [30]:
payload = {
    "criteriaSearch": {
        # "requirement": ["python"]
    }
}

## 4. HTTP headers

A browser-like `User-Agent` plus `Referer`/`Origin` are required for the API to accept the request.

In [31]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Referer": "https://nofluffjobs.com/",
    "Origin": "https://nofluffjobs.com",
}

## 5. Send the request

In [32]:
response = httpx.post(url, params=params, json=payload, headers=headers)
print(f"HTTP status: {response.status_code}")

HTTP status: 200


## 6. Parse and clean the response

On success, extract the `postings` list. Each posting is kept as-is for now; `salary` is read out to confirm it is present.

In [33]:
cleaned_postings = []

if response.status_code == 200:
    data = response.json()
    postings = data.get("postings", [])
    print(f"Success! Retrieved {len(postings)} postings.")

    for job in postings:
        salary = job.get("salary", {})
        cleaned_postings.append(job)
else:
    print(f"Failed with status {response.status_code}: {response.text[:200]}")

Success! Retrieved 107 postings.


## 7. Inspect the collected postings

In [ ]:
# display(cleaned_postings)

[{'id': 'senior-project-manager-ds-transformacji-ai-antal-Warszawa',
  'name': 'Antal',
  'location': {'places': [{'country': {'code': 'POL', 'name': 'Poland'},
     'city': 'Warszawa',
     'street': '',
     'postalCode': '',
     'geoLocation': {'latitude': 52.2296756, 'longitude': 21.0122287},
     'url': 'senior-project-manager-ds-transformacji-ai-antal-warszawa'}],
   'fullyRemote': False,
   'covidTimeRemotely': False,
   'hybridDesc': '',
   'type': 'onsite'},
  'posted': 1793196299520,
  'title': 'Senior Project Manager ds. Transformacji AI',
  'logo': {'original': 'companies/logos/original/antal_international_20230313_115903.png',
   'jobs_details': 'companies/logos/jobs_details/antal_international_20230313_115903.png',
   'jobs_listing': 'companies/logos/jobs_listing/antal_international_20230313_115903.png',
   'jobs_details_2x': 'companies/logos/jobs_details_2x/antal_international_20230313_115903.png',
   'jobs_listing_2x': 'companies/logos/jobs_listing_2x/antal_internation

### Optional: view as a DataFrame

Flatten a few key fields for easier inspection.

In [35]:
if cleaned_postings:
    df = pd.DataFrame([
        {
            "title": j.get("title"),
            "company": j.get("name"),
            "salary_from": j.get("salary", {}).get("from"),
            "salary_to": j.get("salary", {}).get("to"),
            "currency": j.get("salary", {}).get("currency"),
            "remote": j.get("fullyRemote"),
        }
        for j in cleaned_postings
    ])
    display(df)
else:
    print("No postings collected.")

,title,company,salary_from,salary_to,currency,remote
0,Senior Project Manager ds. Transformacji AI,Antal,7560.0,9408.0,EUR,False
1,Technical Leader/Architekt Systemowy,Antal,6720.0,8232.0,EUR,False
2,Senior Java Developer,Antal,5712.0,7224.0,EUR,False
3,Sprzedawca / Sprzedawczyni - praca na stacji p...,Circle K Polska,1008.0,NaN,EUR,False
4,Cyber Security Specialist,Outokumpu,2472.0,3093.0,EUR,False
...,...,...,...,...,...,...
102,Senior Database Administrator,Toyota Digital HUB & SSC (TME NV/SA),4082.0,5670.0,EUR,False
103,Senior Database Administrator,Toyota Digital HUB & SSC (TME NV/SA),4082.0,5670.0,EUR,False
104,Senior Database Administrator,Toyota Digital HUB & SSC (TME NV/SA),4082.0,5670.0,EUR,False
105,Infrastructure Architect Senior Specialist,Toyota Digital HUB & SSC (TME NV/SA),4082.0,5670.0,EUR,False


## 8. Persist to SQLite

Store each posting in a local SQLite database.

In [36]:
import sqlite3
import json as _json

db_path = "job_postings.db"
conn = sqlite3.connect(db_path)

# Flatten every field of each posting into columns, keeping all of them
df_all = pd.json_normalize(cleaned_postings)
if "id" in df_all.columns:
    df_all = df_all.set_index("id")

# Make nested values (lists/dicts) storable in SQLite
df_all = df_all.map(lambda x: _json.dumps(x) if isinstance(x, (list, dict)) else x)

# Create the table if it does not exist yet
cols = ["id"] + list(df_all.columns)
col_defs = ", ".join(["id TEXT PRIMARY KEY"] + [f'"{c}"' for c in df_all.columns])
conn.execute(f"CREATE TABLE IF NOT EXISTS nofluff ({col_defs})")

# Upsert: insert new rows, keep older records on conflict
placeholders = ", ".join(["?"] * len(cols))
col_names = ", ".join([f'"{c}"' for c in cols])
sql = f"INSERT OR IGNORE INTO nofluff ({col_names}) VALUES ({placeholders})"
for idx, row in df_all.iterrows():
    values = [idx] + [row[c] for c in df_all.columns]
    conn.execute(sql, values)
conn.commit()

cur = conn.cursor()
print(f"Saved postings; total in nofluff: {cur.execute('SELECT COUNT(*) FROM nofluff').fetchone()[0]}")
conn.close()


Saved postings; total in nofluff: 426
